# BasaLens — Balinese vs Indonesian AI Translation Audit

**Track:** AI + Languages

**Research question:** Using the same pretrained NLLB model and the same held-out sentences, is English→Balinese translation quality lower than English→Indonesian, and does beam search help?

**Original contribution:** (1) focus on Balinese as the local, lower-resource language, (2) compare greedy decoding with beam search, (3) review a sample with a fluent Balinese speaker, and (4) test original school/community notice examples.

**Important:** This is a research prototype, not a certified translation service. Do not rely on it for medical, legal, emergency, or other high-stakes material. NLLB's model card says it is intended for research and not production deployment. Cite the model, dataset, tools, and ChatGPT assistance in your submission.

## Setup checklist
1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU** (or another available GPU).
3. Sign in to Hugging Face, open `https://huggingface.co/datasets/openlanguagedata/flores_plus`, and accept the dataset access conditions.
4. Create a Hugging Face read token. The login cell below will prompt you securely; do not paste your token into a public cell or GitHub.
5. Run the cells from top to bottom. This notebook uses the FLORES+ `devtest` evaluation split; it does not train on FLORES+.

Dataset license: FLORES+ CC BY-SA 4.0. Model license: NLLB-200 distilled 600M CC-BY-NC 4.0. Review both terms before reuse.

In [ ]:
!pip -q install -U "transformers<5" datasets sacrebleu accelerate sentencepiece gradio

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import CHRF
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Secure Hugging Face login. A hidden prompt will ask for your access token.
# First accept FLORES+ dataset conditions on its Hugging Face page.
login()

## 1. Load aligned evaluation data

FLORES+ provides aligned sentences across languages. We load English, Indonesian, and Balinese `devtest` rows, join them by sentence ID, and select 100 fixed examples. Each English sentence has a professional reference translation in both target languages. We use the benchmark as evaluation data, not training data.

Language codes: `eng_Latn` = English, `ind_Latn` = Indonesian, `ban_Latn` = Balinese. Be careful: `ban_Latn` is Balinese, not Banjar.

In [ ]:
DATASET_NAME = 'openlanguagedata/flores_plus'

def load_language_split(config_name: str, output_column: str) -> pd.DataFrame:
    dataset = load_dataset(DATASET_NAME, config_name, split='devtest')
    frame = dataset.select_columns(['id', 'text']).to_pandas()
    return frame.rename(columns={'text': output_column})

english_df = load_language_split('eng_Latn', 'english')
indonesian_df = load_language_split('ind_Latn', 'indonesian_reference')
balinese_df = load_language_split('ban_Latn', 'balinese_reference')

aligned = english_df.merge(indonesian_df, on='id', how='inner', validate='one_to_one')
aligned = aligned.merge(balinese_df, on='id', how='inner', validate='one_to_one')

if len(aligned) < 100:
    raise ValueError(f'Expected at least 100 aligned rows; found {len(aligned)}. Check dataset access/config names.')

# Fixed random sample makes the evaluation reproducible.
df = aligned.sample(n=100, random_state=SEED).sort_values('id').reset_index(drop=True)
print('Aligned devtest rows:', len(aligned))
print('Evaluation examples:', len(df))
display(df[['id', 'english', 'indonesian_reference', 'balinese_reference']].head(3))

## 2. Load the pretrained translation model

We use `facebook/nllb-200-distilled-600M`. The model is already pretrained; this project evaluates it rather than fine-tuning it. A GPU is strongly recommended. If the model cannot load, do not spend hours debugging—use the time-boxed fallback in the project plan.

In [ ]:
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_dtype = torch.float16 if device.type == 'cuda' else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang='eng_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=model_dtype)
model.to(device)
model.eval()

print('Model loaded on:', device)

## 3. Translate with two decoding settings

**Baseline:** greedy decoding (`num_beams=1`).

**Main comparison:** beam search (`num_beams=4`).

The model, prompts, input sentences, target language, and maximum output length stay the same; only the decoding strategy changes. This is a decoding baseline, not a separately trained model. Report both settings honestly even if beam search does not improve the score.

In [ ]:
def translate_many(texts, target_language, num_beams=4, batch_size=8):
    if target_language not in {'ind_Latn', 'ban_Latn'}:
        raise ValueError('Unsupported target language code.')
    tokenizer.src_lang = 'eng_Latn'
    target_id = tokenizer.convert_tokens_to_ids(target_language)
    if target_id is None or target_id == tokenizer.unk_token_id:
        raise ValueError(f'Could not resolve target language token: {target_language}')

    predictions = []
    for start in range(0, len(texts), batch_size):
        batch_texts = [str(x) for x in texts[start:start + batch_size]]
        encoded = tokenizer(
            batch_texts, return_tensors='pt', padding=True,
            truncation=True, max_length=512
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                forced_bos_token_id=target_id,
                num_beams=num_beams,
                max_new_tokens=128
            )
        predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return predictions

# 100 English sentences × 2 target languages × 2 decoding settings = 400 translations.
for target_code, short_name in [('ind_Latn', 'indonesian'), ('ban_Latn', 'balinese')]:
    df[f'{short_name}_greedy'] = translate_many(df['english'].tolist(), target_code, num_beams=1)
    df[f'{short_name}_beam4'] = translate_many(df['english'].tolist(), target_code, num_beams=4)

display(df[['english', 'indonesian_beam4', 'balinese_beam4']].head(5))

## 4. Score with chrF and compare results

chrF measures character n-gram overlap against the reference translation. Higher is better, but **chrF is not accuracy and does not fully measure meaning or cultural appropriateness**. Report the value as a chrF score, not as a percentage of correct translations.

In [ ]:
chrf = CHRF(word_order=0)

comparisons = [
    ('Indonesian', 'indonesian_greedy', 'indonesian_reference', 'Greedy'),
    ('Indonesian', 'indonesian_beam4', 'indonesian_reference', 'Beam search (4)'),
    ('Balinese', 'balinese_greedy', 'balinese_reference', 'Greedy'),
    ('Balinese', 'balinese_beam4', 'balinese_reference', 'Beam search (4)'),
]

score_rows = []
for language, prediction_col, reference_col, method in comparisons:
    score = chrf.corpus_score(
        df[prediction_col].tolist(), [df[reference_col].tolist()]
    ).score
    score_rows.append({'Language': language, 'Method': method, 'chrF': round(score, 2)})

scores = pd.DataFrame(score_rows)
display(scores.pivot(index='Language', columns='Method', values='chrF'))

In [ ]:
# Visual comparison for the notebook/demo.
plot_data = scores.pivot(index='Language', columns='Method', values='chrF')
ax = plot_data.plot(kind='bar', figsize=(8, 4))
ax.set_title('NLLB translation quality on 100 held-out FLORES+ examples')
ax.set_ylabel('chrF score (higher is better)')
ax.set_xlabel('Target language')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)

# Save aggregate results, not a public copy of the FLORES+ source text.
scores.to_csv('basalens_aggregate_scores.csv', index=False)
print('Saved aggregate score table.')

## 5. Show the five hardest examples

Judges should see the English source, the model output, and the professional reference side by side. Explain at least one error in plain language. Do not cherry-pick only successful examples.

In [ ]:
for language, prediction_col, reference_col, score_col in [
    ('Balinese', 'balinese_beam4', 'balinese_reference', 'balinese_beam4_chrf'),
    ('Indonesian', 'indonesian_beam4', 'indonesian_reference', 'indonesian_beam4_chrf')
]:
    df[score_col] = [
        chrf.sentence_score(hyp, [ref]).score
        for hyp, ref in zip(df[prediction_col], df[reference_col])
    ]

    print(f'Five lowest-scoring {language} examples:')
    display(
        df.nsmallest(5, score_col)[
            ['english', prediction_col, reference_col, score_col]
        ].rename(columns={prediction_col: 'NLLB output', reference_col: 'Reference', score_col: 'sentence chrF'})
    )

    print('Discuss whether the problem is meaning, word choice, names/numbers, register, or grammar.')

## 6. Human review (the original contribution)

Ask a fluent Balinese speaker to review 25 randomly selected NLLB Balinese outputs. Obtain permission before using any person's writing or feedback. Show the reviewer the English source and NLLB output first; ask them to score: **0 = meaning wrong/unusable, 1 = partly correct or awkward, 2 = meaning clear and natural**. Ask them to tag errors (meaning, grammar, register, names/numbers, idiom/cultural context).

Use one reviewer only as an exploratory check; do not claim it represents all Balinese speakers or dialects. If no fluent reviewer is available, state that clearly and do not claim human validation.

In [ ]:
review_sample = df.sample(n=min(25, len(df)), random_state=17).sort_values('id')
display(review_sample[['id', 'english', 'balinese_beam4']].reset_index(drop=True))
print('Reviewer rubric: 0 = wrong/unusable; 1 = partly correct/awkward; 2 = clear and natural.')

## 7. Add your own local school/community notice examples

These are original test prompts, not FLORES+ sentences. Ask a fluent speaker to write or verify the Balinese translations with permission. Do not invent the Balinese reference translations yourself if you are not fluent. Compare the NLLB output with the speaker's version and discuss register and local phrasing.

In [ ]:
local_notices = [
    'The school library will be closed on Friday.',
    'Please return borrowed books before the end of the week.',
    'The student meeting will begin at eight in the morning.',
    'Parents are invited to attend the school performance.',
    'Please keep the classroom clean after the activity.',
    'The event has been moved to the school hall because of rain.',
    'Students should arrive fifteen minutes before the ceremony.',
    'Please contact the school office if you have any questions.',
    'The lost-and-found desk is beside the main entrance.',
    'Thank you for helping keep our school community safe.'
]
local_df = pd.DataFrame({'english': local_notices})
local_df['nllb_indonesian'] = translate_many(local_notices, 'ind_Latn', num_beams=4, batch_size=4)
local_df['nllb_balinese'] = translate_many(local_notices, 'ban_Latn', num_beams=4, batch_size=4)
display(local_df)

## 8. Build the interactive demo

The demo takes a short English notice and returns an AI draft in Balinese or Indonesian. It must not show a made-up confidence score. Always display the reminder that a fluent speaker should review important or culturally specific text.

In [ ]:
import gradio as gr

def demo_translate(text, target_language):
    if not text or not text.strip():
        return 'Please enter a sentence first.'
    target_code = {
        'Balinese (Basa Bali)': 'ban_Latn',
        'Indonesian (Bahasa Indonesia)': 'ind_Latn'
    }[target_language]
    return translate_many([text.strip()], target_code, num_beams=4, batch_size=1)[0]

demo = gr.Interface(
    fn=demo_translate,
    inputs=[
        gr.Textbox(lines=4, label='English source text', placeholder='Type a short school or community notice...'),
        gr.Dropdown(
            choices=['Balinese (Basa Bali)', 'Indonesian (Bahasa Indonesia)'],
            value='Balinese (Basa Bali)', label='Translate into'
        )
    ],
    outputs=gr.Textbox(lines=5, label='AI translation draft'),
    title='BasaLens',
    description='A research prototype comparing AI translation into Balinese and Indonesian. Have a fluent speaker review important text.'
)
demo.launch(share=True)

## 9. What to write down before submitting

Fill in the real results from the cells above—never invent or round results to make the model look better.

- Number of evaluation sentences: 100 (or actual number displayed).
- Indonesian chrF: greedy = ___; beam 4 = ___.
- Balinese chrF: greedy = ___; beam 4 = ___.
- Human review: ___ outputs reviewed by ___ fluent Balinese speaker(s); score distribution = ___.
- Biggest observed failure: ___.
- Limitation: FLORES+ sentences are benchmark/news/travel/general text and may not represent everyday Balinese conversation, dialects, or formal registers. NLLB is a research model, not certified translation.

## Sources and credits
- FLORES+: https://huggingface.co/datasets/openlanguagedata/flores_plus (CC BY-SA 4.0; cite the dataset papers listed on its card; use the dataset for evaluation, not training).
- NLLB-200 distilled 600M: https://huggingface.co/facebook/nllb-200-distilled-600M (CC-BY-NC 4.0; research model).
- Libraries: Hugging Face Transformers, Datasets, PyTorch, sacrebleu, pandas, Gradio.
- AI assistance: ChatGPT was used for project planning and code assistance; the team reviewed, ran, and understands the submitted code. Add any other AI tools used.

**Before sharing:** Do not upload your Hugging Face token. Do not publicly redistribute the full FLORES+ text or a copy of its raw dataset files. Share the notebook code and aggregate results, and follow the dataset's access and sharing conditions.